# 03 — Social Media Links Study

**Pipeline step:** Section 3.1.1 of the paper — first look at all URLs extracted from Telegram
messages, before filtering down to YouTube.

**Purpose.**
1. Inspect the most-shared URLs overall.
2. Break down shared links by social platform (X/Twitter, Instagram, YouTube, Facebook,
   TikTok), including which profiles/channels are most shared for X and Instagram.
3. Reproduce **Figure 1** of the paper: number of URL occurrences per month, showing the
   election-month peak.

**Input:**
- `../data/telegram_2024/urls_per_message` — Parquet dataset of unique URLs, each with a list
  of `occurrences` (one entry per time/chat the URL was posted).
- `../data/telegram_2024/urls.json.gz` — the same data in line-delimited JSON, used for the
  monthly aggregation (streamed rather than loaded into Spark, since we only need counts).

**Output:**
- `../reports/Figures/contagem_ocorrencias.png` — Figure 1 of the paper.
- `../data/counter_urls_by_month.json` — monthly URL-occurrence counts, cached for reuse.

**Requires:** a running Spark session (`spark`) — see `04_Basic_Analysis.ipynb` if running
standalone.

**Next step:** `06_Filtering_Yt_Videos.ipynb`, which isolates the YouTube subset.


In [ ]:
import gzip
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import desc, regexp_extract, size
from tqdm import tqdm

spark = SparkSession.builder.appName("TelegramLinksStudy").getOrCreate()

In [ ]:
DATA_DIR = Path("../data")
URLS_PARQUET_PATH = f"{DATA_DIR}/telegram_2024/urls_per_message"
URLS_JSONL_GZ_PATH = f"{DATA_DIR}/telegram_2024/urls.json.gz"
FIGURES_DIR = Path("../reports/Figures")

## 1. Most-shared URLs overall

In [ ]:
df = spark.read.parquet(URLS_PARQUET_PATH)
df = df.withColumn("occurrences_count", size("occurrences"))
df.select("url", "occurrences", "occurrences_count").createOrReplaceTempView("view_links")

df.show()

In [ ]:
def top_by_occurrences(sql_df, n: int = 20):
    """Print the n URLs with the most occurrences in `sql_df`."""
    sql_df.select("url", "occurrences_count").orderBy(desc("occurrences_count")).show(n, truncate=False)


print("Top 20 URLs overall by number of occurrences:")
top_by_occurrences(df)

## 2. Breakdown by platform

For each platform we report the top shared URLs and the total number of links. For X/Twitter
and Instagram we additionally extract the profile handle from the URL (the segment right after
`.com/`) to see which accounts are most frequently shared.

In [ ]:
PLATFORM_PATTERNS = {
    "X/Twitter": "url LIKE '%.x.%' OR url LIKE '%.twitter.%' OR url LIKE '%t.co%'",
    "Instagram": "url LIKE '%.instagram.%'",
    "YouTube": "url LIKE '%.youtube.%' OR url LIKE '%youtu.be%'",
    "Facebook": "url LIKE '%.facebook.%' OR url LIKE '%.fb.%' OR url LIKE '%.fb.watch%'",
    "TikTok": "url LIKE '%.tiktok.%'",
}


def platform_links(pattern: str, label: str, n: int = 20):
    """Return the subset of `view_links` matching `pattern`, printing its top URLs and total count."""
    subset = spark.sql(f"SELECT * FROM view_links WHERE {pattern}")
    print(f"Top {n} most-shared {label} URLs:")
    top_by_occurrences(subset, n)
    print(f"Total {label} links: {subset.count()}\n")
    return subset


def top_profiles(platform_df, n: int = 20):
    """Rank profiles/accounts (the segment right after '.com/' in the URL) by total occurrences."""
    profiles = platform_df.withColumn("profile", regexp_extract("url", r"\.com/([^/?]+)", 1))
    profiles.select("profile", "occurrences_count").createOrReplaceTempView("view_profile")
    return spark.sql("""
        SELECT profile, SUM(occurrences_count) AS total
        FROM view_profile
        GROUP BY profile
        ORDER BY total DESC
    """)


platform_dfs = {
    label: platform_links(pattern, label) for label, pattern in PLATFORM_PATTERNS.items()
}

In [ ]:
print("Most-shared X/Twitter profiles:")
top_profiles(platform_dfs["X/Twitter"]).show(20)

print("Most-shared Instagram profiles:")
top_profiles(platform_dfs["Instagram"]).show(20)

In [ ]:
# Preview of YouTube links pointing to channel pages rather than individual videos —
# these are excluded from the video-level analysis in the next notebook (Section 3.1.3).
spark.sql("""
    SELECT url, occurrences_count
    FROM view_links
    WHERE url LIKE '%.youtube.%' AND url LIKE '%channel%'
    ORDER BY occurrences_count DESC
""").show(truncate=False)

## 3. URL occurrences per month (Figure 1)

We stream the line-delimited JSON version of the dataset (rather than using Spark) since we
only need a running count of occurrences per month, keyed by each occurrence's `date` field.

In [ ]:
monthly_counts = Counter()

with gzip.open(URLS_JSONL_GZ_PATH, "rt", encoding="utf-8") as f:
    for line in tqdm(f, desc="Processing URL records"):
        if not line.strip():
            continue
        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            continue
        for occurrence in item.get("occurrences", []):
            if "date" in occurrence:
                month = occurrence["date"][:7]  # YYYY-MM
                monthly_counts[month] += 1

DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(DATA_DIR / "counter_urls_by_month.json", "w") as f:
    json.dump(monthly_counts, f)

In [ ]:
# Shared plot style used across the paper's figures.
sns.set_theme(style="white", font="Liberation Sans")
plt.rcParams.update({
    "font.family": "Liberation Sans",
    "font.size": 26,
    "axes.titlesize": 26,
    "axes.labelsize": 26,
    "xtick.labelsize": 26,
    "ytick.labelsize": 26,
    "legend.fontsize": 26,
    "figure.titlesize": 26,
    "text.color": "#3F3F3FD8",
    "axes.labelcolor": "#3F3F3FD8",
    "xtick.color": "#3F3F3FD8",
    "ytick.color": "#3F3F3FD8",
    "font.weight": 700,
    "axes.labelweight": 700,
    "axes.titleweight": 700,
})

In [ ]:
months_ordered = sorted(monthly_counts.keys())
values_ordered = [monthly_counts[m] for m in months_ordered]

fig, ax = plt.subplots(figsize=(12, 7))
ax.bar(months_ordered, values_ordered, color="#4a4a4a", width=0.8)

ax.set_axisbelow(True)
ax.grid(True, which="major", linestyle="-", linewidth=0.75, alpha=0.55)
ax.minorticks_on()
ax.grid(True, which="minor", linestyle="-", linewidth=0.25, alpha=0.45)

ax.set_ylabel("Occurrences")
ax.set_xlabel("Month-Year")
ax.set_ylim(0, max(values_ordered) * 1.1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIGURES_DIR / "contagem_ocorrencias.png", format="png", dpi=300, bbox_inches="tight")
plt.show()